# Tiny Mitra — guided Colab training
Prepared locally; actual Colab execution is pending. This notebook trains our manual PyTorch models from random weights on the final saved corpus of1,000 synthetic examples, using fixed task pieces and character fallback. Review the actual GPU pilot before the4,000-update main budget. Earlier character/continued-training experiments are separate history, not initialization for the final models.

## 1. Upload the project bundle
In Colab use Runtime → Change runtime type and choose an available GPU. The next cell opens a file picker: choose `tiny-mitra-cloud.zip` produced by tools/prepare_notebooks.py. Only project source, tests and synthetic data are bundled.

In [ ]:
from google.colab import files
from pathlib import Path
import os, zipfile, hashlib, json, sys, subprocess
uploaded = files.upload()
archive = next(name for name in uploaded if name.endswith('.zip'))
root = Path('/content/tiny-mitra').resolve()
root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as bundle:
    for member in bundle.namelist():
        assert (root / member).resolve().is_relative_to(root), 'Archive path escapes project'
    bundle.extractall(root)
os.chdir(root)
manifest = json.loads(Path('bundle_manifest.json').read_text())
for name, expected in manifest.items():
    assert hashlib.sha256(Path(name).read_bytes()).hexdigest() == expected, name
print('Project bundle verified:', len(manifest), 'files')

## 2. Dependencies and actual GPU check
This verifies allocated hardware rather than assuming that selecting GPU succeeded. Package installation may take a few minutes.

In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-r','requirements.txt'],check=True)
import torch
print('Python:',sys.version,'PyTorch:',torch.__version__)
assert torch.cuda.is_available(), 'No GPU allocated. Reconnect to an available GPU or discuss the CPU/alternative plan.'
print('GPU:',torch.cuda.get_device_name(0))
device='cuda'

## 3. Correctness and data
Passing these checks is necessary before training. The GPU smoke check explicitly places model and IDs on CUDA.

In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q'],check=True)
DATASET_PATH='artifacts/mitra_final/dataset/examples.json'
assert Path(DATASET_PATH).exists(),'Use the current project bundle containing the synthetic corpus'
from broad_data import validate_broad
print(validate_broad(json.loads(Path(DATASET_PATH).read_text())))
from model import TinyGPT
from config import ModelConfig
from task_tokenizer import TaskTokenizer
tokenizer=TaskTokenizer()
smoke=TinyGPT(ModelConfig(len(tokenizer),context=256)).to(device)
print('GPU logits shape:',smoke(torch.tensor([[2,4,12]],device=device)).shape)
del smoke
subprocess.run([sys.executable,'-m','tools.pilot','--device',device,'--output','artifacts/checkpoints/cloud_tiny_pilot','--report','artifacts/results/cloud_tiny_pilot.json'],check=True)

## 4. Authorize Drive and restore saved work
The permission dialog is yours to approve. Runtime disk is temporary; Drive stores completed checkpoint copies. The notebook only uses `MyDrive/TinyMitra` for this project.

In [ ]:
from google.colab import drive
import shutil,time
drive.mount('/content/drive')
persistent=Path('/content/drive/MyDrive/TinyMitra')
persistent.mkdir(exist_ok=True)
if (persistent/'artifacts').exists():
    shutil.copytree(persistent/'artifacts',Path('artifacts'),dirs_exist_ok=True)
print('Restored available artifacts; run identity is checked by train.py')

## 5. Timed GPU pilot and interruption/resume
Run 100 updates first. The second command resumes to a TOTAL of 200 updates, not another 200. Review the recorded validation curves and elapsed time before choosing the main comparison budget. This is a pilot, not a final result.

In [ ]:
pilot=['--architecture','transformer_2','--device',device,'--dataset',DATASET_PATH,'--output','artifacts/task_cloud_pilot','--validate-every','50']
subprocess.run([sys.executable,'task_train.py',*pilot,'--steps','100','--resume'],check=True)
subprocess.run([sys.executable,'task_train.py',*pilot,'--steps','200','--resume'],check=True)
print(Path('artifacts/task_cloud_pilot/history.json').read_text())

## 6. Main runs with durable checkpoint copies
Set STEPS only after the pilot review. An interrupted run resumes using the same configuration/source/data. Keep the runtime open for interactive work; ordinary Colab availability/session limits still apply.

In [ ]:
def persist_artifacts():
    for source in Path('artifacts').rglob('*'):
        if source.is_file() and source.suffix != '.tmp':
            target=persistent/source
            target.parent.mkdir(parents=True,exist_ok=True)
            temporary=target.with_name(target.name+'.copying')
            shutil.copy2(source,temporary)
            temporary.replace(target)
STEPS=4000  # Freeze after real GPU pilot review; this is a separate broad-only study.
from config import ARCHITECTURES
from task_runtime import task_source_hash
from data import digest
cloud_records=json.loads(Path(DATASET_PATH).read_text())
cloud_protocol={'steps':STEPS,'architectures':list(ARCHITECTURES),'seed':42,'dataset_hash':digest(cloud_records),'source_hash':task_source_hash(),'device':device,'gpu':torch.cuda.get_device_name(0),'selection':'fixed final update; best loss saved separately','initialization':'random weights on the same frozen final corpus; local and cloud artifacts stay separate'}
protocol_path=Path('artifacts/task_tokens_cloud/results/protocol.json')
protocol_path.parent.mkdir(parents=True,exist_ok=True)
if protocol_path.exists():
    assert json.loads(protocol_path.read_text())==cloud_protocol,'Changed protocol: use a separately reviewed directory'
else:
    protocol_path.write_text(json.dumps(cloud_protocol,indent=2))
for architecture in ARCHITECTURES:
    command=[sys.executable,'task_train.py','--architecture',architecture,'--device',device,'--dataset',DATASET_PATH,'--steps',str(STEPS),'--output',f'artifacts/task_tokens_cloud/checkpoints/{architecture}_seed42','--resume']
    process=subprocess.Popen(command)
    while process.poll() is None:
        time.sleep(30)
        persist_artifacts()
    persist_artifacts()
    assert process.returncode==0,'Run failed; inspect output and saved state'

## 7. Development evaluation and export
The cloud main run starts from random weights on the same final synthetic corpus and declared4,000-update budget. It is unverified until actually executed; hardware/software differences and recovery still need checks. Authored/validation diagnostics are not final-test performance. After download/extraction, use `python app.py --task-root artifacts/task_tokens_cloud` while preserving local results.

In [ ]:
subprocess.run([sys.executable,'-m','tools.evaluate_task','--device',device,'--dataset',DATASET_PATH],check=True)
subprocess.run([sys.executable,'-m','tools.portability','--write-reference','--device',device,'--tokenizer-kind','task','--checkpoint','artifacts/task_tokens_cloud/checkpoints/transformer_2_seed42/latest.pt','--reference','artifacts/task_tokens_cloud/results/portability_reference.json'],check=True)
persist_artifacts()
archive=shutil.make_archive('/content/tiny-mitra-trained','zip','artifacts')
files.download(archive)

## Remaining acceptance evidence
Record real GPU/version/timing. After a saved checkpoint, disconnect and restore in a fresh runtime; compare with an uninterrupted control. Two commands in one runtime alone do not prove runtime-loss recovery. On your laptop run `python -m tools.portability --tokenizer-kind task --checkpoint artifacts/task_tokens_cloud/checkpoints/transformer_2_seed42/latest.pt --reference artifacts/task_tokens_cloud/results/portability_reference.json`. Check actual gpu_to_cpu_passed, logit tolerance and greedy continuation. Learning, cloud verification and final-test release remain separate gates.